# Week 12 Live Coding: Where Should the Last \$50 Million Go?

Seven states, 93 electoral votes, and you need 44 of them. Today we:

1. Simulate 200,000 elections and read your chance of reaching 270 off the picture.
2. Let each state's polls miss independently, and watch your chance drop.
3. See where the analytics director's 0.07 came from.
4. Build diminishing returns, and run the first round of the algorithm by hand.
5. Run all fifty rounds, once for each goal.
6. Rerun the plan at the top and bottom of her interval.

### Before you start

**There is no problem set this week.** Before you leave, print this notebook to PDF and upload it to the **in-class** assignment on Canvas (the steps are at the bottom). That is your attendance credit. It is not graded on being right.

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy. Work in that tab; edits to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the data straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup

Two datasets. `states` has one row per swing state: its electoral votes and your margin in the polling average. `campaigns` is the analytics director's data, which we use in Part 3.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

states = pd.read_csv(
    'https://raw.githubusercontent.com/joshuakalla/'
    'data_science_campaigns_26/main/weeks/'
    'wk12_presidential_allocation/data/swing_states.csv')
campaigns = pd.read_csv(
    'https://raw.githubusercontent.com/joshuakalla/'
    'data_science_campaigns_26/main/weeks/'
    'wk12_presidential_allocation/data/ad_spending_effects.csv')

states

You already hold 226 electoral votes. The simulation needs each state's electoral votes and margin as plain lists of seven numbers.

In [ ]:
held = 226                                  # electoral votes you hold

# .values: just the seven numbers, without the table around them
state_evs = states['electoral_votes'].values
margin = states['margin'].values

print('electoral votes:', state_evs)
print('your margin:', margin)

## Part 1: Simulate the election

Polls miss in two ways at once. A **national** miss hits every state in the same direction (in 2020 the polls overstated Biden almost everywhere). A **state** miss is different in each state.

The cell below does steps 1 to 4 from the slides for all 200,000 elections at once. Each row is one simulated election, and each column is one state.

In [ ]:
np.random.seed(2026)        # same seed as earlier weeks
n_elections = 200000

# np.random.normal(average, SD, (rows, columns)) makes random draws

# Step 1: one national miss per election, SD 3.5. It is a single
# column, so every state gets the same national miss.
national_miss = np.random.normal(0, 3.5, (n_elections, 1))

# Step 2: a different miss in each of the seven states, SD 2.0
state_miss = np.random.normal(0, 2.0, (n_elections, 7))

# Step 3: your poll margin plus both misses
result = margin + national_miss + state_miss

# Step 4: True where you won the state. Add up your electoral votes.
won = result > 0
your_evs = held + (won * state_evs).sum(axis=1)  # axis=1: across a row

# the first eight elections as a table: your result in each state
elections = pd.DataFrame(result.round(1), columns=states['state'])
elections['your EVs'] = your_evs
elections.head(8)

*Check: the "your EVs" column reads 241, 226, 241, 260, 319, 226, 226, 226.*

These are random draws, so a few numbers in this notebook differ slightly from the slides. The conclusions do not.

Now count across all 200,000.

In [ ]:
print('average electoral votes:', round(your_evs.mean(), 1))
print('chance of reaching 270:',
      round(100 * (your_evs >= 270).mean(), 1), '%')

*Check: 261.8 and 39.5%.*

You never get 261.8 electoral votes. Here is what you do get.

In [ ]:
plt.hist(your_evs, bins=50, color='gray')
plt.axvline(270, color='#B58900')   # the 270 line
plt.xlabel('Your electoral votes')
plt.ylabel('Number of simulated elections')
plt.show()

The two tall bars are losing all seven states (226) and winning all seven (319). That is the national miss: when the polls are wrong, they are wrong in every state at once.

## Part 2: What if states missed independently?

Give each state the same total SD, 4.03 points, but let every state miss on its own.

In [ ]:
# spread of both misses together: square, add, take the root
total_sd = np.sqrt(3.5**2 + 2.0**2)
print('total SD in one state:', round(total_sd, 2))

# new draws: each state misses on its own, SD 4.03
np.random.seed(33)
indep_miss = np.random.normal(0, total_sd, (n_elections, 7))
won_indep = (margin + indep_miss) > 0
evs_indep = held + (won_indep * state_evs).sum(axis=1)

print('misses move together:',
      round(100 * (your_evs >= 270).mean(), 1), '%')
print('misses independent:',
      round(100 * (evs_indep >= 270).mean(), 1), '%')

*Check: 4.03, then 39.5% and 33.4%.*

You are behind in five of the seven states. With independent misses you need five separate misses in your favor. With a shared miss, one national miss in your favor helps you in all five at once.

## Part 3: Where the analytics director's 0.07 came from

Sixty past campaigns, one row each: how many million dollars more the campaign spent on ads in a state than its opponent, and how many points its margin in that state moved.

In [ ]:
campaigns.head(10)

In [ ]:
reg = smf.ols('margin_shift_pp ~ spend_advantage_m',
              data=campaigns).fit()
coef = reg.params['spend_advantage_m']

# conf_int() gives the 95% interval: [0] is the low end, [1] the high end
ci = reg.conf_int().loc['spend_advantage_m']
lo = ci[0]
hi = ci[1]

print('$1M moves a state', round(coef, 3), 'points')
print('95% interval:', round(lo, 3), 'to', round(hi, 3))

*Check: 0.071, and 0.033 to 0.109.*

**Nobody randomized any of this.** Campaigns choose where to spend. If they put more money where they were already gaining, this line credits the ads for gains they did not cause. The interval does not fix that: it covers sampling error only.

We build the plan on 0.07 anyway. Part 6 asks what happens if it is wrong.

## Part 4: Diminishing returns, and the first million by hand

We assume no amount of money moves one state more than 40 × 0.07 = 2.8 points (we made this up). The formula below gets there slowly: each million buys a little less than the one before.

In [ ]:
# np.exp(x) is e to the power x. As d grows, np.exp(-d / 40) shrinks
# toward 0, so the margin bought rises toward 40 * 0.07 = 2.8 and stops.
# 0.07 is coef rounded, so these match the slides.
for d in [1, 10, 25, 50]:
    bought = 40 * 0.07 * (1 - np.exp(-d / 40))
    print('$' + str(d) + 'M buys', round(bought, 2), 'points.',
          'A straight line says', round(0.07 * d, 2))

*Check: \$50M buys 2.0 points, against 3.5 on a straight line.*

Now round 1 of the algorithm. For each state: put \$1M there and nowhere else, rerun the same 200,000 elections, and see how much your chance of reaching 270 goes up.

In [ ]:
before = (your_evs >= 270).mean()     # 39.5%, nothing spent yet
gains = []

for i in range(7):
    spend = np.zeros(7)               # $M in each state: none yet
    spend[i] = 1                      # $1M in state i only
    new_margin = margin + 40 * coef * (1 - np.exp(-spend / 40))
    won_new = (new_margin + national_miss + state_miss) > 0
    evs_new = held + (won_new * state_evs).sum(axis=1)
    gain = 100 * ((evs_new >= 270).mean() - before)
    gains.append(round(gain, 3))

states['first $1M'] = gains           # points of chance of 270
states

*Check: Pennsylvania +0.255, Nevada +0.034.*

Pennsylvania comes first: big and close. Nevada is closer than Georgia, Arizona and North Carolina and comes last, because six electoral votes rarely decide the election.

## Part 5: The allocator, with two goals

Fifty rounds. Each round does what the cell above did, then gives the million to the state where it raised your chance of 270 the most.

In [ ]:
plan_270 = np.zeros(7)            # $M given to each state so far

for r in range(50):               # fifty rounds of $1M
    scores = []
    for i in range(7):
        extra = np.zeros(7)
        extra[i] = 1              # try one more $1M in state i
        spend = plan_270 + extra
        new_margin = margin + 40 * coef * (1 - np.exp(-spend / 40))
        won_new = (new_margin + national_miss + state_miss) > 0
        evs_new = held + (won_new * state_evs).sum(axis=1)
        scores.append((evs_new >= 270).mean())   # chance of 270
    best = np.argmax(scores)      # argmax: position of the biggest score
    plan_270[best] = plan_270[best] + 1

states['plan_270'] = plan_270.astype(int)   # astype(int): whole $M
states

*Check: PA 34, MI 7, WI 9.*

Now the same loop with one line changed: score each try by **expected electoral votes**, the average across the 200,000 elections, instead of the chance of reaching 270.

In [ ]:
plan_ev = np.zeros(7)

for r in range(50):
    scores = []
    for i in range(7):
        extra = np.zeros(7)
        extra[i] = 1
        spend = plan_ev + extra
        new_margin = margin + 40 * coef * (1 - np.exp(-spend / 40))
        won_new = (new_margin + national_miss + state_miss) > 0
        evs_new = held + (won_new * state_evs).sum(axis=1)
        scores.append(evs_new.mean())   # <- the only line that changed
    best = np.argmax(scores)
    plan_ev[best] = plan_ev[best] + 1

states['plan_ev'] = plan_ev.astype(int)
states

*Check: PA 22, GA 11, NC 7, MI 10.*

Score both plans on both goals, on the same 200,000 elections.

In [ ]:
for plan in [plan_ev, plan_270]:
    spend = plan
    new_margin = margin + 40 * coef * (1 - np.exp(-spend / 40))
    won_new = (new_margin + national_miss + state_miss) > 0
    evs_new = held + (won_new * state_evs).sum(axis=1)
    print('expected EVs', round(evs_new.mean(), 1),
          '| chance of 270', round(100 * (evs_new >= 270).mean(), 1), '%')

*Check: 266.5 and 46.5% for the expected-EV plan, then 266.1 and 47.6% for the chance-of-270 plan.*

The expected-EV plan wins on expected electoral votes and loses the election more often.

The chance-of-270 plan bought Pennsylvania 19, Michigan 15 and Wisconsin 10. **That is 44, exactly the number you need.** Expected electoral votes counts a vote you did not need the same as one you did, so that plan buys votes in Georgia and North Carolina that mostly arrive in elections you were already winning.

## Part 6: What if the effect is smaller, or zero?

Keep the chance-of-270 plan and change only how much \$1M moves a state: the top of her interval, her estimate, and the bottom.

In [ ]:
for c in [hi, coef, lo]:
    new_margin = margin + 40 * c * (1 - np.exp(-plan_270 / 40))
    won_new = (new_margin + national_miss + state_miss) > 0
    evs_new = held + (won_new * state_evs).sum(axis=1)
    print('if $1M moves a state', round(c, 3), 'points: chance of 270',
          round(100 * (evs_new >= 270).mean(), 1), '%')

# at zero the money moves nothing, so you stay where you started
print('if it moves nothing: chance of 270', round(100 * before, 1), '%')

*Check: 52.0%, 47.6%, 43.1%, 39.5%.*

\$50 million buys somewhere between nothing and 12.5 points of win probability (52.0 − 39.5). The experiments in Kalla and Broockman (2018) cannot rule out zero for ads like these.

---

## Before you leave

1. **Runtime → Restart session and run all.** It runs the notebook from top to bottom, in order. The allocator cells take a few seconds each.
2. **Check that every cell ran.** Scroll from the top. Every code cell should show its output below it.
3. **File → Print → Save as PDF.**
4. **Upload the PDF** to the **in-class** assignment on Canvas.